# Análise de Carteira de Ações B3

Notebook de análise técnica e fundamentalista para uma carteira de 10 ações da B3.

**Carteira:** BBAS3 · ABEV3 · B3SA3 · GGBR4 · ITSA4 · PETR4 · RENT3 · SUZB3 · VALE3 · WEGE3

---
**Seções:**
1. Setup e Configuração
2. Dados de Cotações (OHLCV)
3. Dados Fundamentalistas
4. Visualizações Técnicas (Candlestick, MAs, Retorno Normalizado)
5. Visualizações Fundamentalistas (Graham, P/L, ROE, DY)
6. Resumo da Carteira

## 1. Setup e Configuração

In [ ]:
# Instalar dependências (executar apenas se necessário)
# !pip install yfinance fundamentus mplfinance pandas numpy matplotlib seaborn

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import mplfinance as mpf
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
import fundamentus

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='darkgrid', palette='muted')

print('Bibliotecas carregadas.')

In [ ]:
# Configuração da carteira
CARTEIRA_YF = [
    'BBAS3.SA', 'ABEV3.SA', 'B3SA3.SA', 'GGBR4.SA', 'ITSA4.SA',
    'PETR4.SA', 'RENT3.SA', 'SUZB3.SA', 'VALE3.SA', 'WEGE3.SA',
]
# Fundamentus não usa sufixo .SA
CARTEIRA_FUND = [t.replace('.SA', '') for t in CARTEIRA_YF]

START_DATE = '2024-06-01'
END_DATE   = '2026-06-06'

print(f'Carteira: {CARTEIRA_FUND}')
print(f'Período : {START_DATE} → {END_DATE}')

## 2. Dados de Cotações (OHLCV)

In [ ]:
# Download de 2 anos de dados históricos via Yahoo Finance
# auto_adjust=True ajusta preços por splits e dividendos automaticamente
df_raw = yf.download(
    CARTEIRA_YF,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True,
)
print(f'Dados: {df_raw.shape[0]} pregões × {len(CARTEIRA_YF)} ativos')

In [ ]:
# Transformar MultiIndex (Price × Ticker) → formato tidy
# future_stack=True suprime FutureWarning do pandas 4
cotacoes = df_raw.stack(level=1, future_stack=True).reset_index()
cotacoes.rename(columns={'Ticker': 'Ativo', 'Date': 'Data'}, inplace=True)
cotacoes['Ativo'] = cotacoes['Ativo'].str.replace('.SA', '', regex=False)

# Volume incluído (necessário para análise técnica)
cotacoes = cotacoes[['Data', 'Ativo', 'Open', 'High', 'Low', 'Close', 'Volume']]
cotacoes.sort_values(['Ativo', 'Data'], inplace=True)
cotacoes.reset_index(drop=True, inplace=True)

print(f'cotacoes: {cotacoes.shape}')
cotacoes.tail()

In [ ]:
# Calcular Médias Móveis por ativo
for janela in [20, 50, 200]:
    cotacoes[f'MA{janela}'] = (
        cotacoes.groupby('Ativo')['Close']
        .transform(lambda s: s.rolling(janela, min_periods=1).mean())
        .round(2)
    )

cotacoes[['Data', 'Ativo', 'Close', 'MA20', 'MA50', 'MA200']].tail(10)

## 3. Dados Fundamentalistas

In [ ]:
# Buscar dados de cada ativo via fundamentus.get_papel()
COLS = [
    'Setor', 'Cotacao', 'Min_52_sem', 'Max_52_sem', 'Valor_de_mercado',
    'Nro_Acoes', 'Patrim_Liq', 'Receita_Liquida_12m', 'Receita_Liquida_3m',
    'Lucro_Liquido_12m', 'Lucro_Liquido_3m',
]

def get_papel_safe(ticker):
    return fundamentus.get_papel(ticker).reindex(columns=COLS)

ind = pd.concat([get_papel_safe(t) for t in CARTEIRA_FUND])
ind = ind.reset_index().rename(columns={'index': 'Ativo'})
ind[[c for c in COLS if c != 'Setor']] = (
    ind[[c for c in COLS if c != 'Setor']].apply(pd.to_numeric, errors='coerce')
)
ind.head(3)

In [ ]:
# Buscar P/L, P/VP, ROE e Dividend Yield
# CORREÇÃO DO BUG ORIGINAL: get_resultado_raw() usa tickers SEM sufixo .SA
# CARTEIRA_FUND já está correto (sem .SA)
resultado = fundamentus.get_resultado_raw().reset_index()
resultado.rename(columns={'papel': 'Ativo', 'Div.Yield': 'DY'}, inplace=True)

ind_2 = resultado[resultado['Ativo'].isin(CARTEIRA_FUND)][['Ativo','P/L','DY','P/VP','ROE']].copy()
ind_2.reset_index(drop=True, inplace=True)
ind_2

In [ ]:
# Unir os dois DataFrames
indicadores = pd.merge(ind, ind_2, on='Ativo', how='left')

# LPA = Lucro por Ação (usa Lucro Líquido, NÃO Receita)
indicadores['LPA'] = (indicadores['Lucro_Liquido_12m'] / indicadores['Nro_Acoes']).round(2)
indicadores['VPA'] = (indicadores['Patrim_Liq']        / indicadores['Nro_Acoes']).round(2)

# Fórmula de Graham: VI = √(22,5 × LPA × VPA)
def graham_vi(row):
    lpa, vpa = row['LPA'], row['VPA']
    if pd.notna(lpa) and pd.notna(vpa) and lpa > 0 and vpa > 0:
        return round(math.sqrt(22.5 * lpa * vpa), 2)
    return None

indicadores['Graham_VI']       = indicadores.apply(graham_vi, axis=1)
indicadores['Graham_Upside_%'] = (
    (indicadores['Graham_VI'] - indicadores['Cotacao']) / indicadores['Cotacao'] * 100
).round(1)
indicadores['DY_%']            = (indicadores['DY'] * 100).round(2)

indicadores[['Ativo','Cotacao','P/L','P/VP','ROE','DY_%','LPA','VPA','Graham_VI','Graham_Upside_%']]

## 4. Visualizações Técnicas

In [ ]:
# Candlestick com volume — últimos 90 pregões de cada ativo
mpf_style = mpf.make_mpf_style(base_mpf_style='nightclouds', rc={'font.size': 9})

for ticker in CARTEIRA_FUND:
    sub = cotacoes[cotacoes['Ativo'] == ticker].tail(90).copy()
    sub = sub.set_index('Data')[['Open','High','Low','Close','Volume']]
    sub.index = pd.DatetimeIndex(sub.index)
    mpf.plot(
        sub, type='candle', volume=True,
        title=f'  {ticker} — Candlestick (últimos 90 pregões)',
        style=mpf_style, figratio=(14, 5), tight_layout=True,
    )

In [ ]:
# Retorno Normalizado (Base 100) — compara todos os ativos na mesma escala
pivot = cotacoes.pivot(index='Data', columns='Ativo', values='Close')
norm  = (pivot / pivot.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 6))
norm.plot(ax=ax, linewidth=1.5)
ax.axhline(100, color='white', lw=0.8, ls='--', alpha=0.4, label='Base (100)')
ax.set_title('Retorno Normalizado da Carteira (Base 100)', fontsize=14)
ax.set_xlabel('Data')
ax.set_ylabel('Retorno Normalizado')
ax.legend(loc='upper left', ncol=5, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Preço de Fechamento com Médias Móveis — PETR4 como exemplo
ticker_ex = 'PETR4'
df_ex = cotacoes[cotacoes['Ativo'] == ticker_ex]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_ex['Data'], df_ex['Close'], color='#cdd6f4', lw=1,   label='Fechamento')
ax.plot(df_ex['Data'], df_ex['MA20'],  color='#ffa726', lw=1.2, label='MA 20')
ax.plot(df_ex['Data'], df_ex['MA50'],  color='#ab47bc', lw=1.2, label='MA 50')
ax.plot(df_ex['Data'], df_ex['MA200'], color='#42a5f5', lw=1.2, label='MA 200')
ax.set_title(f'{ticker_ex} — Preço com Médias Móveis', fontsize=13)
ax.set_xlabel('Data')
ax.set_ylabel('Preço (R$)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Visualizações Fundamentalistas

In [ ]:
# Scatter: Valor Intrínseco Graham vs Cotação Atual
df_g = indicadores.dropna(subset=['Graham_VI']).copy()

fig, ax = plt.subplots(figsize=(10, 7))
lim = max(df_g[['Cotacao','Graham_VI']].max()) * 1.15
ax.plot([0, lim], [0, lim], 'r--', lw=1.2, label='VI = Cotação (justo)', alpha=0.7)

cores = ['#26a69a' if v > c else '#ef5350'
         for v, c in zip(df_g['Graham_VI'], df_g['Cotacao'])]
ax.scatter(df_g['Cotacao'], df_g['Graham_VI'], s=120, c=cores,
           zorder=3, edgecolors='white', lw=0.5)

for _, r in df_g.iterrows():
    ax.annotate(r['Ativo'], (r['Cotacao'], r['Graham_VI']),
                textcoords='offset points', xytext=(7, 4), fontsize=9, color='white')

ax.set_xlabel('Cotação Atual (R$)', fontsize=11)
ax.set_ylabel('VI Graham (R$)', fontsize=11)
ax.set_title('VI = √(22,5 × LPA × VPA)\nVerde = subavaliado  ·  Vermelho = sobreavaliado', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Upside Graham por ativo
df_u = indicadores.dropna(subset=['Graham_Upside_%']).sort_values('Graham_Upside_%')

cores_u = ['#26a69a' if v > 15 else '#ffca28' if v >= -15 else '#ef5350'
           for v in df_u['Graham_Upside_%']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_u['Ativo'], df_u['Graham_Upside_%'], color=cores_u, alpha=0.9)
ax.axvline(0,   color='white',   lw=0.8, ls='--', alpha=0.7)
ax.axvline(15,  color='#26a69a', lw=0.8, ls=':',  alpha=0.7, label='Margem +15%')
ax.axvline(-15, color='#ef5350', lw=0.8, ls=':',  alpha=0.7, label='Margem −15%')
for bar, v in zip(bars, df_u['Graham_Upside_%']):
    ax.text(v + 0.4 if v >= 0 else v - 0.4,
            bar.get_y() + bar.get_height() / 2,
            f'{v:+.1f}%', va='center',
            ha='left' if v >= 0 else 'right', fontsize=9)
ax.set_xlabel('Upside / Downside em relação ao VI Graham (%)')
ax.set_title('Upside Graham por Ativo', fontsize=12)
ax.legend(fontsize=8)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Comparativo de P/L, ROE e Dividend Yield
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
df_m = indicadores.sort_values('Ativo').dropna(subset=['P/L'])

for ax, (col, title, cor) in zip(axes, [
    ('P/L',  'P/L (Preço / Lucro)',   '#4fc3f7'),
    ('ROE',  'ROE (% sobre PL)',      '#26a69a'),
    ('DY_%', 'Dividend Yield (%)',    '#ffa726'),
]):
    vals = df_m[col] * 100 if col == 'ROE' else df_m[col]
    ax.barh(df_m['Ativo'], vals, color=cor, alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.grid(True, axis='x', alpha=0.3)
    for i, v in enumerate(vals):
        if pd.notna(v):
            ax.text(v + vals.max() * 0.01, i, f'{v:.1f}', va='center', fontsize=8)

plt.suptitle('Indicadores Fundamentalistas da Carteira', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Resumo da Carteira

In [ ]:
# Tabela resumo estilizada com Pandas Styler
cols_resumo = [
    'Ativo','Setor','Cotacao','Min_52_sem','Max_52_sem',
    'P/L','P/VP','ROE','DY_%','LPA','VPA','Graham_VI','Graham_Upside_%',
]
resumo = indicadores[cols_resumo].copy()

def color_upside(v):
    if pd.isna(v): return ''
    if v > 15:     return 'background-color:#1a3a2a;color:#26a69a'
    if v < -15:    return 'background-color:#3a1a1a;color:#ef5350'
    return 'background-color:#2a2a1a;color:#ffca28'

(
    resumo.style
    .format({
        'Cotacao':         'R$ {:.2f}',
        'Min_52_sem':      'R$ {:.2f}',
        'Max_52_sem':      'R$ {:.2f}',
        'P/L':             '{:.1f}',
        'P/VP':            '{:.2f}',
        'ROE':             '{:.1%}',
        'DY_%':            '{:.2f}%',
        'LPA':             'R$ {:.2f}',
        'VPA':             'R$ {:.2f}',
        'Graham_VI':       lambda v: f'R$ {v:.2f}' if pd.notna(v) else '—',
        'Graham_Upside_%': lambda v: f'{v:+.1f}%' if pd.notna(v) else '—',
    }, na_rep='—')
    .background_gradient(subset=['P/L'], cmap='RdYlGn_r', low=0, high=1)
    .background_gradient(subset=['ROE'], cmap='RdYlGn',   low=0, high=1)
    .applymap(color_upside, subset=['Graham_Upside_%'])
    .set_caption('Resumo Fundamentalista da Carteira B3')
)